# 🔬 SOTA Skin Lesion Classification ('3-Brain' Hybrid Architecture)
### Target: 96-97% Diagnostic Accuracy on 7-Class HAM10000 Dataset
**DenseNet-201 (Local) + CBAM (Attention) + ViT-B/16 (Global Self-Attention) + Multiscale Fusion**


---
## 📦 Block 1: Environment Setup & Hardware Verification
Installs required packages and verifies NVIDIA GPU acceleration (T4 / A100 / V100).


In [ ]:
# 1. Install dependencies
!pip install -q albumentations opencv-python-headless scikit-learn seaborn kagglehub tqdm

import os
import glob
import shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from typing import Optional, Tuple, Dict, List

# Hardware verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Hardware] Active PyTorch Device: {device}")
if torch.cuda.is_available():
    print(f"[Hardware] GPU Accelerator: {torch.cuda.get_device_name(0)}")
    print(f"[Hardware] GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[WARNING] Running on CPU. For fast training, enable GPU via Runtime -> Change runtime type -> T4 GPU.")


---
## 🧪 Block 2: Clinical Preprocessing (DullRazor + CLAHE in CIE-LAB)
1. **DullRazor Morphological Inpainting:** Isolates hair artifacts via Black-Hat morphological filtering and restores skin textures with Telea inpainting.
2. **CIE-LAB CLAHE Equalization:** Balances lighting variations strictly on the $L$ (Luminance) channel while preserving clinical chrominance ($A, B$).


In [ ]:
"""
=============================================================================
Phase 1: Advanced Preprocessing Pipeline & Data Augmentation
=============================================================================
Author: Lead AI Computer Vision Engineer
Project: SOTA Skin Lesion Classification (HAM10000)

Pipeline:
1. DullRazor Morphological Hair Removal (Black-Hat Filtering + Masking + Inpainting)
2. Lighting Correction via CLAHE on LAB Color Space (Preserves Color Fidelity)
3. Geometric & Color Augmentations via Albumentations
4. Standardization: Resize (224x224) & ImageNet Normalization
=============================================================================
"""

import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ImageNet statistics for transfer learning backbones
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def apply_dull_razor(image: np.ndarray, filter_size: int = 9, threshold: int = 10, inpaint_radius: int = 5) -> np.ndarray:
    """
    Morphological hair removal algorithm inspired by DullRazor.
    
    Workflow:
    1. Grayscale conversion.
    2. Black-Hat morphological filtering to isolate dark, narrow hair filaments.
    3. Binary thresholding to generate hair mask.
    4. Telea inpainting to reconstruct skin surface behind detected hairs.
    
    Args:
        image (np.ndarray): Input RGB image of shape (H, W, 3) with uint8 values [0, 255].
        filter_size (int): Structuring element kernel size for morphological filter.
        threshold (int): Intensity difference threshold for hair mask generation.
        inpaint_radius (int): Pixel radius for Fast Marching Method inpainting.
        
    Returns:
        np.ndarray: Hair-removed RGB image.
    """
    if image is None or image.size == 0:
        raise ValueError("Invalid input image passed to apply_dull_razor.")
    
    # 1. Convert to grayscale for morphological filtering
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    
    # 2. Black-Hat filter highlights structures darker than their surroundings
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (filter_size, filter_size))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    
    # 3. Threshold to obtain hair mask
    _, hair_mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)
    
    # Optional morphological closing on mask to bridge small gaps in detected hairs
    mask_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    hair_mask = cv2.morphologyEx(hair_mask, cv2.MORPH_CLOSE, mask_kernel)
    
    # 4. Inpainting to restore skin texture underneath hairs
    hair_removed = cv2.inpaint(image, hair_mask, inpaintRadius=inpaint_radius, flags=cv2.INPAINT_TELEA)
    return hair_removed


def apply_clahe(image: np.ndarray, clip_limit: float = 2.0, tile_grid_size: tuple = (8, 8)) -> np.ndarray:
    """
    Contrast Limited Adaptive Histogram Equalization (CLAHE) in CIE-LAB color space.
    
    Applying CLAHE directly in RGB causes severe unnatural color shifts, which alters 
    diagnostic features like erythema (redness) and melanin pigmentation. By transforming
    to CIE-LAB and equalizing ONLY the L (Luminance) channel, we enhance local edge 
    and texture contrast while strictly preserving clinical chrominance (A & B channels).
    
    Args:
        image (np.ndarray): Input RGB image [0, 255].
        clip_limit (float): Threshold for contrast limiting.
        tile_grid_size (tuple): Grid dimension for contextual regions (default: 8x8).
        
    Returns:
        np.ndarray: Contrast-enhanced RGB image.
    """
    if image is None or image.size == 0:
        raise ValueError("Invalid input image passed to apply_clahe.")
    
    # Convert RGB to CIE-LAB
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    
    # Apply CLAHE to L channel
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    cl = clahe.apply(l_channel)
    
    # Merge channels and convert back to RGB
    enhanced_lab = cv2.merge((cl, a_channel, b_channel))
    enhanced_rgb = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2RGB)
    return enhanced_rgb


def preprocess_lesion_image(image: np.ndarray, img_size: int = 224) -> np.ndarray:
    """
    Full clinical preprocessing pipeline combining DullRazor + CLAHE.
    
    Args:
        image (np.ndarray): Raw dermoscopy RGB image.
        img_size (int): Target square dimension.
        
    Returns:
        np.ndarray: Cleaned, contrast-adjusted, resized RGB image (uint8, 0-255).
    """
    # Step 1: DullRazor hair removal
    hair_free = apply_dull_razor(image, filter_size=9, threshold=10, inpaint_radius=5)
    
    # Step 2: Lighting correction via CLAHE
    enhanced = apply_clahe(hair_free, clip_limit=2.0, tile_grid_size=(8, 8))
    
    # Step 3: Resize
    if enhanced.shape[0] != img_size or enhanced.shape[1] != img_size:
        enhanced = cv2.resize(enhanced, (img_size, img_size), interpolation=cv2.INTER_CUBIC)
        
    return enhanced


def get_transforms(img_size: int = 224):
    """
    Builds Albumentations transformation pipelines for training and validation/testing.
    
    Training pipeline features heavy spatial and photometric augmentations to artificially
    inflate minority classes and avoid overfitting on benign nevi.
    
    Args:
        img_size (int): Image spatial resolution (224 for DenseNet/ViT).
        
    Returns:
        tuple: (train_transform, val_transform)
    """
    train_transform = A.Compose([
        A.Resize(img_size, img_size, interpolation=cv2.INTER_CUBIC),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.1,
            scale_limit=0.15,
            rotate_limit=45,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.7
        ),
        A.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.15,
            hue=0.05,
            p=0.5
        ),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=(3, 5), p=1.0),
        ], p=0.3),
        A.CoarseDropout(
            num_holes_range=(1, 4),
            hole_height_range=(12, 28),
            hole_width_range=(12, 28),
            fill=0,
            p=0.4
        ),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])
    
    val_transform = A.Compose([
        A.Resize(img_size, img_size, interpolation=cv2.INTER_CUBIC),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])
    
    return train_transform, val_transform


---
## 📂 Block 3: HAM10000 Automated Dataset Ingestion & Zero-Leakage Split
- Automatically downloads the HAM10000 dataset via `kagglehub`.
- Performs **Lesion-ID Stratified Partitioning** guaranteeing 0% data leakage across patients.
- Creates class-balanced DataLoaders with Albumentations data augmentation.


In [ ]:
"""
=============================================================================
Phase 1 & 3: HAM10000 Dataset Loader, Automated Ingestion & Split Management
=============================================================================
Author: Lead AI Computer Vision Engineer
Project: SOTA Skin Lesion Classification (HAM10000)

Features:
- Automated dataset acquisition via kagglehub / kaggle API / fallback.
- Lesion-level Stratified Splitting to eliminate data leakage between train/val/test.
- DullRazor + CLAHE integrated directly into PyTorch Dataset pipeline.
- WeightedRandomSampler support to actively equalize mini-batch class frequencies.
=============================================================================
"""

import os
import glob
import shutil
import zipfile
import numpy as np
import pandas as pd
from PIL import Image
from typing import Optional, Tuple, Dict, List

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedGroupKFold, train_test_split



# Canonical HAM10000 7 Diagnostic Classes Mapping
LABEL_TO_IDX: Dict[str, int] = {
    'akiec': 0,  # Actinic keratoses and intraepithelial carcinoma
    'bcc':   1,  # Basal cell carcinoma
    'bkl':   2,  # Benign keratosis-like lesions
    'df':    3,  # Dermatofibroma
    'mel':   4,  # Melanoma
    'nv':    5,  # Melanocytic nevi
    'vasc':  6   # Vascular lesions
}

IDX_TO_LABEL: Dict[int, str] = {v: k for k, v in LABEL_TO_IDX.items()}

CLASS_FULL_NAMES: Dict[str, str] = {
    'akiec': 'Actinic Keratoses / Bowen\'s Disease',
    'bcc':   'Basal Cell Carcinoma',
    'bkl':   'Benign Keratosis-like Lesions',
    'df':    'Dermatofibroma',
    'mel':   'Melanoma',
    'nv':    'Melanocytic Nevi',
    'vasc':  'Vascular Lesions'
}


def download_and_extract_ham10000(dest_dir: str = "./data/ham10000") -> str:
    """
    Automated acquisition and extraction pipeline for the HAM10000 dataset.
    
    Checks local existence first; if missing, triggers download via kagglehub
    or official Kaggle CLI.
    
    Args:
        dest_dir (str): Root destination directory.
        
    Returns:
        str: Absolute path to the resolved dataset root.
    """
    os.makedirs(dest_dir, exist_ok=True)
    metadata_candidates = glob.glob(os.path.join(dest_dir, "**", "*metadata*.csv"), recursive=True)
    
    if metadata_candidates:
        print(f"[Dataset] HAM10000 metadata discovered locally: {metadata_candidates[0]}")
        return dest_dir

    print("[Dataset] HAM10000 not found locally. Initiating automated acquisition...")
    
    # Strategy 1: kagglehub (Google/Kaggle official lightweight downloader)
    try:
        import kagglehub
        print("[Dataset] Attempting download via kagglehub...")
        download_path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
        print(f"[Dataset] Successfully downloaded to cache: {download_path}")
        
        # Link or copy into dest_dir
        for item in os.listdir(download_path):
            s = os.path.join(download_path, item)
            d = os.path.join(dest_dir, item)
            if not os.path.exists(d):
                if os.path.isdir(s):
                    shutil.copytree(s, d)
                else:
                    shutil.copy2(s, d)
        print(f"[Dataset] Assets staged in: {dest_dir}")
        return dest_dir
    except Exception as e:
        print(f"[Dataset] kagglehub acquisition skipped/failed: {e}")

    # Strategy 2: Kaggle CLI API
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        print("[Dataset] Attempting download via Kaggle CLI API...")
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files("kmader/skin-cancer-mnist-ham10000", path=dest_dir, unzip=True)
        print(f"[Dataset] Kaggle CLI download and extraction complete: {dest_dir}")
        return dest_dir
    except Exception as e:
        print(f"[Dataset] Kaggle API failed: {e}")
        print("\n" + "="*70)
        print("MANUAL DOWNLOAD INSTRUCTION:")
        print("1. Download 'skin-cancer-mnist-ham10000' from Kaggle or Harvard Dataverse:")
        print("   https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000")
        print(f"2. Unzip all contents into: {os.path.abspath(dest_dir)}")
        print("="*70 + "\n")
        return dest_dir


def prepare_ham10000_dataframe(data_dir: str) -> pd.DataFrame:
    """
    Indexes all image files, merges with HAM10000 metadata, and creates a clean DataFrame.
    
    Args:
        data_dir (str): Root dataset folder.
        
    Returns:
        pd.DataFrame: Clean DataFrame with columns [image_id, lesion_id, dx, path, label].
    """
    metadata_files = glob.glob(os.path.join(data_dir, "**", "*metadata*.csv"), recursive=True)
    if not metadata_files:
        raise FileNotFoundError(f"Could not locate HAM10000_metadata.csv inside {data_dir}")
    
    meta_df = pd.read_csv(metadata_files[0])
    
    # Map all image files in all subdirectories (part 1, part 2, etc.)
    all_image_paths = glob.glob(os.path.join(data_dir, "**", "*.jpg"), recursive=True)
    if not all_image_paths:
        raise FileNotFoundError(f"No .jpg images found inside {data_dir}")
    
    image_path_map = {
        os.path.splitext(os.path.basename(p))[0]: p
        for p in all_image_paths
    }
    
    meta_df['path'] = meta_df['image_id'].map(image_path_map)
    # Drop records where image file is not present
    meta_df = meta_df.dropna(subset=['path']).reset_index(drop=True)
    
    # Encode target label integers
    meta_df['label'] = meta_df['dx'].map(LABEL_TO_IDX)
    meta_df = meta_df.dropna(subset=['label']).reset_index(drop=True)
    meta_df['label'] = meta_df['label'].astype(int)
    
    print(f"[Dataset] Indexed {len(meta_df)} dermoscopy images across 7 diagnostic classes.")
    return meta_df


class HAM10000Dataset(Dataset):
    """
    PyTorch Dataset for HAM10000 with DullRazor + CLAHE preprocessing & Albumentations.
    """
    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform=None,
        apply_clinical_preprocess: bool = True,
        img_size: int = 224
    ):
        """
        Args:
            dataframe (pd.DataFrame): Dataframe containing 'path' and 'label'.
            transform: Albumentations Compose pipeline.
            apply_clinical_preprocess (bool): Whether to run DullRazor + CLAHE.
            img_size (int): Image size to format into.
        """
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.apply_clinical_preprocess = apply_clinical_preprocess
        self.img_size = img_size

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = row['path']
        label = row['label']

        # Load RGB image via PIL -> np.ndarray
        img = Image.open(img_path).convert('RGB')
        img_np = np.array(img)

        # Apply Phase 1 Preprocessing: DullRazor hair removal + CLAHE
        if self.apply_clinical_preprocess:
            img_np = preprocess_lesion_image(img_np, img_size=self.img_size)

        # Apply Phase 1/3 Albumentations augmentations / normalization
        if self.transform is not None:
            augmented = self.transform(image=img_np)
            tensor_img = augmented['image']
        else:
            # Fallback simple tensor conversion
            tensor_img = torch.from_numpy(img_np.transpose((2, 0, 1))).float() / 255.0

        return tensor_img, torch.tensor(label, dtype=torch.long), row['image_id']


def create_stratified_splits(
    df: pd.DataFrame,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    random_state: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Performs Patient/Lesion-aware Stratified Splitting.
    
    Crucial for dermatological integrity: If the same lesion was photographed multiple 
    times, putting one in Train and one in Test artificially inflates test accuracy.
    This function groups by lesion_id while preserving class proportions.
    """
    # Group by lesion_id and obtain dominant class per lesion
    lesion_df = df.groupby('lesion_id').agg({'dx': 'first', 'label': 'first'}).reset_index()
    
    # Check if stratified splitting is feasible (test set size must be >= num_classes and min class count >= 2)
    n_classes = len(lesion_df['label'].unique())
    min_class_count = lesion_df['label'].value_counts().min()
    can_stratify_test = (int(len(lesion_df) * test_ratio) >= n_classes) and (min_class_count >= 2)
    stratify_target = lesion_df['label'] if can_stratify_test else None

    # First split: Train+Val vs Test
    train_val_lesions, test_lesions = train_test_split(
        lesion_df,
        test_size=test_ratio,
        stratify=stratify_target,
        random_state=random_state
    )
    
    # Second split: Train vs Val
    val_adjusted_ratio = val_ratio / (1.0 - test_ratio)
    n_tv_classes = len(train_val_lesions['label'].unique())
    can_stratify_val = (int(len(train_val_lesions) * val_adjusted_ratio) >= n_tv_classes) and (train_val_lesions['label'].value_counts().min() >= 2)
    stratify_train_val = train_val_lesions['label'] if can_stratify_val else None
    train_lesions, val_lesions = train_test_split(
        train_val_lesions,
        test_size=val_adjusted_ratio,
        stratify=stratify_train_val,
        random_state=random_state
    )
    
    train_df = df[df['lesion_id'].isin(train_lesions['lesion_id'])].reset_index(drop=True)
    val_df   = df[df['lesion_id'].isin(val_lesions['lesion_id'])].reset_index(drop=True)
    test_df  = df[df['lesion_id'].isin(test_lesions['lesion_id'])].reset_index(drop=True)
    
    # Strict Mathematical Verification: Zero Lesion or Image Leakage
    train_lesions_set = set(train_df['lesion_id'])
    val_lesions_set = set(val_df['lesion_id'])
    test_lesions_set = set(test_df['lesion_id'])
    
    train_images_set = set(train_df['image_id'])
    val_images_set = set(val_df['image_id'])
    test_images_set = set(test_df['image_id'])
    
    assert train_lesions_set.isdisjoint(test_lesions_set), (
        f"[DATA LEAKAGE ERROR] Train and Test share {len(train_lesions_set.intersection(test_lesions_set))} lesions!"
    )
    assert train_lesions_set.isdisjoint(val_lesions_set), (
        f"[DATA LEAKAGE ERROR] Train and Val share {len(train_lesions_set.intersection(val_lesions_set))} lesions!"
    )
    assert val_lesions_set.isdisjoint(test_lesions_set), (
        f"[DATA LEAKAGE ERROR] Val and Test share {len(val_lesions_set.intersection(test_lesions_set))} lesions!"
    )
    assert train_images_set.isdisjoint(test_images_set), "[DATA LEAKAGE ERROR] Image overlap between Train and Test!"
    
    print("=" * 75)
    print("           PATIENT/LESION-STRATIFIED SPLIT INTEGRITY VERIFIED")
    print("=" * 75)
    print(f"Total Lesions:  {len(lesion_df)} (Train: {len(train_lesions_set)}, Val: {len(val_lesions_set)}, Test: {len(test_lesions_set)})")
    print(f"Total Images:   {len(df)} (Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)})")
    print(f"Disjoint Check: Zero lesion overlap between Train, Val, and Test sets (Strictly 0%).")
    print("=" * 75)
    return train_df, val_df, test_df


def build_dataloaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    batch_size: int = 32,
    num_workers: int = 2,
    img_size: int = 224,
    use_weighted_sampler: bool = True
) -> Tuple[DataLoader, DataLoader, DataLoader, torch.Tensor]:
    """
    Constructs PyTorch DataLoaders with Albumentations pipelines and optional 
    WeightedRandomSampler for minority oversampling.
    
    Returns:
        tuple: (train_loader, val_loader, test_loader, class_weights)
    """
    train_transform, val_transform = get_transforms(img_size=img_size)

    train_dataset = HAM10000Dataset(train_df, transform=train_transform, apply_clinical_preprocess=True, img_size=img_size)
    val_dataset   = HAM10000Dataset(val_df, transform=val_transform, apply_clinical_preprocess=True, img_size=img_size)
    test_dataset  = HAM10000Dataset(test_df, transform=val_transform, apply_clinical_preprocess=True, img_size=img_size)

    # Calculate class distribution
    class_counts = train_df['label'].value_counts().sort_index().values
    total_samples = len(train_df)
    class_weights = total_samples / (len(class_counts) * class_counts.astype(np.float32))
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

    # Setup mini-batch sampler to equalize minority classes during training
    sampler = None
    shuffle = True
    if use_weighted_sampler:
        sample_weights = [class_weights[label] for label in train_df['label']]
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        shuffle = False

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, val_loader, test_loader, class_weights_tensor


---
## 🧠 Block 4: The '3-Brain' Hybrid Deep Learning Architecture
1. **Brain 1 (Local):** DenseNet-201 capturing dense micro-textures (1920-d).
2. **Brain 2 (Attention):** CBAM (Channel & Spatial Attention) focusing on lesions.
3. **Brain 3 (Global Context):** Vision Transformer (ViT-B/16) modeling long-range non-local patch dependencies via [CLS] token (768-d).
4. **Multiscale Fusion:** $1920 + 768 = 2688$-d fused representation with deep regularized MLP head.


In [ ]:
"""
=============================================================================
Phase 2: The "3-Brain" Hybrid Architecture (DenseNet201 + CBAM + ViT + Fusion)
=============================================================================
Author: Lead AI Computer Vision Engineer
Project: SOTA Skin Lesion Classification (HAM10000)

Architecture Breakdown:
-----------------------------------------------------------------------------
1. Brain 1 (Local Feature Extractor):
   - Pretrained DenseNet-201 backbone extracting fine-grained multi-scale local 
     features, lesion textures, borders, and color variations.
   - Output Feature Map: [Batch, 1920, 7, 7] (for 224x224 input)

2. Brain 2 (Attention Block - CBAM):
   - Convolutional Block Attention Module placed directly after DenseNet-201.
   - Channel Attention Module (CAM): Recalibrates channel-wise feature responses
     via shared MLP on both AvgPool and MaxPool representations.
   - Spatial Attention Module (SAM): Highlights lesion regions and suppresses 
     healthy skin/vignetting artifacts via 7x7 spatial convolution.
   - Global Pooling -> 1x1920 refined local feature vector.

3. Brain 3 (Global Context Extractor):
   - Pretrained Vision Transformer (ViT-B/16) operating in parallel.
   - 12-layer Multi-Head Self-Attention capturing long-range global contextual 
     dependencies across the entire dermoscopic lesion.
   - Extracts 1x768 Global Feature Vector from the [CLS] token.

4. Feature Fusion & Decision Head:
   - Concatenates Local (1920) + Global (768) -> 1x2688 Fused Representation.
   - Deep Regularized Classification Head with Batch Normalization, Dropout, 
     GELU activations, and 7-class projection.
=============================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import DenseNet201_Weights, ViT_B_16_Weights


# ===========================================================================
# 1. Convolutional Block Attention Module (CBAM)
# ===========================================================================
class ChannelAttention(nn.Module):
    """
    Channel Attention Module (CAM) of CBAM.
    
    Exploits inter-channel relationships of features by using both Global Average 
    Pooling and Global Max Pooling followed by a shared Multi-Layer Perceptron (MLP).
    """
    def __init__(self, in_planes: int, reduction_ratio: int = 16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        reduced_planes = max(in_planes // reduction_ratio, 16)
        
        # Shared MLP implemented via 1x1 Convolutions
        self.mlp = nn.Sequential(
            nn.Conv2d(in_planes, reduced_planes, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced_planes, in_planes, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        channel_weights = self.sigmoid(avg_out + max_out)
        return x * channel_weights


class SpatialAttention(nn.Module):
    """
    Spatial Attention Module (SAM) of CBAM.
    
    Exploits inter-spatial relationship of features by applying channel-wise pooling
    (Avg + Max) concatenated along channel axis, followed by a 7x7 convolution.
    """
    def __init__(self, kernel_size: int = 7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), "Kernel size must be 3 or 7"
        padding = 3 if kernel_size == 7 else 1
        
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        scale = torch.cat([avg_out, max_out], dim=1)
        spatial_weights = self.sigmoid(self.conv(scale))
        return x * spatial_weights


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module combining sequential Channel & Spatial Attention.
    """
    def __init__(self, in_planes: int, reduction_ratio: int = 16, kernel_size: int = 7):
        super(CBAM, self).__init__()
        self.channel_att = ChannelAttention(in_planes, reduction_ratio=reduction_ratio)
        self.spatial_att = SpatialAttention(kernel_size=kernel_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x


# ===========================================================================
# 2. "3-Brain" Hybrid Network
# ===========================================================================
class ThreeBrainHybridLesionClassifier(nn.Module):
    """
    State-of-the-Art Hybrid Classifier combining:
    - DenseNet-201 (1920-d local texture extractor)
    - CBAM (Channel & Spatial Attention refinement)
    - ViT-B/16 (768-d global contextual self-attention extractor)
    - Multi-scale Fusion Head (1920 + 768 = 2688 -> 7 classes)
    """
    def __init__(
        self,
        num_classes: int = 7,
        pretrained: bool = True,
        dropout_rate: float = 0.4,
        freeze_backbones: bool = False
    ):
        super(ThreeBrainHybridLesionClassifier, self).__init__()
        self.num_classes = num_classes

        # -------------------------------------------------------------------
        # Brain 1: DenseNet-201 Local Feature Extractor
        # -------------------------------------------------------------------
        dense_weights = DenseNet201_Weights.DEFAULT if pretrained else None
        base_densenet = models.densenet201(weights=dense_weights)
        self.densenet_features = base_densenet.features
        self.densenet_out_dim = 1920  # DenseNet201 last layer feature dimension

        # -------------------------------------------------------------------
        # Brain 2: CBAM Attention Block
        # -------------------------------------------------------------------
        self.cbAM = CBAM(in_planes=self.densenet_out_dim, reduction_ratio=16, kernel_size=7)
        self.local_pool = nn.AdaptiveAvgPool2d((1, 1))

        # -------------------------------------------------------------------
        # Brain 3: ViT-B/16 Global Context Extractor
        # -------------------------------------------------------------------
        vit_weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        self.vit = models.vit_b_16(weights=vit_weights)
        self.vit_out_dim = 768  # ViT-B/16 hidden dimension
        
        # Replace classification heads in ViT with Identity to extract raw [CLS] token embedding
        self.vit.heads = nn.Identity()

        if freeze_backbones:
            self._freeze_backbone_weights()

        # -------------------------------------------------------------------
        # Feature Fusion & Classification Head
        # Fused dimension: 1920 (DenseNet+CBAM) + 768 (ViT) = 2688
        # -------------------------------------------------------------------
        self.fused_dim = self.densenet_out_dim + self.vit_out_dim  # 2688
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.fused_dim),
            nn.Dropout(p=dropout_rate),
            nn.Linear(self.fused_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(p=dropout_rate * 0.75),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(p=dropout_rate * 0.5),
            nn.Linear(128, num_classes)
        )

        # Cache for Grad-CAM gradients and activations
        self.gradients = None
        self.activations = None

    def _freeze_backbone_weights(self):
        """Freezes backbone weights for warm-up phase if desired."""
        for param in self.densenet_features.parameters():
            param.requires_grad = False
        for param in self.vit.parameters():
            param.requires_grad = False

    def unfreeze_backbones(self):
        """Unfreezes all parameters for full end-to-end fine-tuning."""
        for param in self.parameters():
            param.requires_grad = True

    def forward_features(self, x: torch.Tensor):
        """
        Extracts local attention features and global ViT features.
        
        Args:
            x (torch.Tensor): Input batch [B, 3, 224, 224]
            
        Returns:
            tuple: (local_vector [B, 1920], global_vector [B, 768], attention_map [B, 1920, 7, 7])
        """
        # 1. DenseNet201 local features
        dense_feats = self.densenet_features(x)  # [B, 1920, 7, 7]
        
        # 2. CBAM Attention refinement
        refined_feats = self.cbAM(dense_feats)   # [B, 1920, 7, 7]
        self.activations = refined_feats         # Save for Grad-CAM
        
        local_vec = self.local_pool(refined_feats)  # [B, 1920, 1, 1]
        local_vec = torch.flatten(local_vec, 1)     # [B, 1920]

        # 3. ViT Global context features ([CLS] token representation)
        global_vec = self.vit(x)                    # [B, 768]

        return local_vec, global_vec, refined_feats

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        End-to-end forward pass.
        
        Args:
            x (torch.Tensor): Input batch [B, 3, 224, 224]
            
        Returns:
            torch.Tensor: Unnormalized logits [B, 7]
        """
        local_vec, global_vec, _ = self.forward_features(x)
        
        # 4. Feature Fusion: [B, 1920 + 768] = [B, 2688]
        fused = torch.cat([local_vec, global_vec], dim=1)
        
        # 5. Classification
        logits = self.classifier(fused)
        return logits


def build_model(num_classes: int = 7, pretrained: bool = True, dropout_rate: float = 0.4) -> ThreeBrainHybridLesionClassifier:
    """Convenience factory function."""
    return ThreeBrainHybridLesionClassifier(
        num_classes=num_classes,
        pretrained=pretrained,
        dropout_rate=dropout_rate
    )


---
## ⚖️ Block 5: Multi-Class Focal Loss & Optimization Engine
- **Class-Weighted Focal Loss ($\gamma = 2.0$):** Suppresses majority class gradients (nevi) and boosts learning on under-represented malignancies (melanoma, dermatofibroma).
- **AdamW + Differential Learning Rates:** $1e-5$ for backbones, $1e-4$ for CBAM attention and fusion head.
- **Cosine Annealing & Early Stopping:** Monitors validation Macro-F1 score to prevent overfitting.


In [ ]:
"""
=============================================================================
Phase 3: Handling Class Imbalance - Multi-Class Focal Loss
=============================================================================
Author: Lead AI Computer Vision Engineer
Project: SOTA Skin Lesion Classification (HAM10000)

Mathematical Formulation:
-----------------------------------------------------------------------------
Standard Cross-Entropy treats all classes and error magnitudes equally:
    CE(p_t) = -log(p_t)

In HAM10000, ~67% of cases are Melanocytic Nevi ('nv'). Easy negative examples 
dominate the gradient, suppressing learning on critical malignant cases 
(e.g., Melanoma 'mel', Basal Cell Carcinoma 'bcc', Actinic Keratosis 'akiec').

Focal Loss introduces:
1. Modulating Factor: (1 - p_t)^gamma
   - When an easy sample has high confidence (p_t -> 1), (1 - p_t)^gamma -> 0,
     diminishing its loss contribution.
   - When a difficult sample has low confidence (p_t -> 0), (1 - p_t)^gamma -> 1,
     preserving full gradient magnitude.
2. Alpha Weighting: alpha_t
   - Inversely balances class frequencies across the 7 dermatological categories.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
=============================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Union, List


class FocalLoss(nn.Module):
    """
    Multi-class Focal Loss with class-weighting and numerical stabilization.
    """
    def __init__(
        self,
        alpha: Optional[Union[float, List[float], torch.Tensor]] = None,
        gamma: float = 2.0,
        reduction: str = 'mean',
        label_smoothing: float = 0.0
    ):
        """
        Args:
            alpha (Tensor, list, or float, optional): Class weights alpha_t. If tensor or list, 
                                                    must match number of classes.
            gamma (float): Focusing parameter gamma >= 0 (default: 2.0).
            reduction (str): 'mean', 'sum', or 'none'.
            label_smoothing (float): Label smoothing coefficient in [0, 1].
        """
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.label_smoothing = label_smoothing

        if alpha is not None:
            if isinstance(alpha, (list, tuple)):
                alpha = torch.tensor(alpha, dtype=torch.float32)
            elif isinstance(alpha, (int, float)):
                alpha = torch.tensor([alpha], dtype=torch.float32)
            self.register_buffer('alpha', alpha)
        else:
            self.alpha = None

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Forward computation.
        
        Args:
            inputs (torch.Tensor): Unnormalized predicted logits of shape (B, C).
            targets (torch.Tensor): Ground-truth class indices of shape (B,).
            
        Returns:
            torch.Tensor: Computed Focal Loss.
        """
        # Compute log probabilities and probabilities with numerical stability
        log_p = F.log_softmax(inputs, dim=1)  # (B, C)
        p = torch.exp(log_p)                  # (B, C)

        # Gather probability corresponding to target class
        targets = targets.view(-1, 1)         # (B, 1)
        log_pt = log_p.gather(1, targets)     # (B, 1)
        pt = p.gather(1, targets)             # (B, 1)

        # Compute focal weighting factor: (1 - p_t)^gamma
        focal_weight = torch.pow(1.0 - pt, self.gamma)

        # Apply class balancing alpha if provided
        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            at = self.alpha.gather(0, targets.squeeze(1)).view(-1, 1)
            loss = -at * focal_weight * log_pt
        else:
            loss = -focal_weight * log_pt

        # Apply label smoothing regularization if enabled
        if self.label_smoothing > 0.0:
            smoothed_loss = -log_p.mean(dim=1, keepdim=True)
            loss = (1.0 - self.label_smoothing) * loss + self.label_smoothing * smoothed_loss

        # Reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss.squeeze(1)


def compute_effective_num_class_weights(class_counts: list, beta: float = 0.9999) -> torch.Tensor:
    """
    Computes class weights based on the 'Class-Balanced Loss Based on Effective Number of Samples'
    (Cui et al., CVPR 2019):
        E_n = (1 - beta^n) / (1 - beta)
        weight_c = 1 / E_n_c
    
    Args:
        class_counts (list or np.ndarray): Number of samples per class.
        beta (float): Hyperparameter in [0, 1). Default 0.9999.
        
    Returns:
        torch.Tensor: Normalized weight tensor of shape (num_classes,).
    """
    class_counts = torch.tensor(class_counts, dtype=torch.float32)
    effective_num = 1.0 - torch.pow(beta, class_counts)
    weights = (1.0 - beta) / (effective_num + 1e-8)
    
    # Normalize weights so sum equals num_classes
    weights = weights / weights.sum() * len(class_counts)
    return weights

"""
=============================================================================
Phase 4: Training Engine, Optimization & Early Stopping by Macro-F1
=============================================================================
Author: Lead AI Computer Vision Engineer
Project: SOTA Skin Lesion Classification (HAM10000)

Key Optimization Strategies:
-----------------------------------------------------------------------------
1. Optimizer: AdamW with weight decay (0.01) and differential learning rates:
   - 1e-5 for pre-trained backbones (DenseNet-201, ViT-B/16)
   - 1e-4 for newly initialized CBAM attention and Fusion/Classifier Head
2. LR Scheduler: CosineAnnealingLR for smooth convergence to global minima.
3. Loss: Class-Weighted Focal Loss (gamma=2.0) mitigating extreme class skew.
4. Model Selection Criterion: Macro-F1 score with Early Stopping (patience=10).
5. Mixed Precision: Automatic Mixed Precision (AMP) for maximum GPU throughput.
=============================================================================
"""

import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm
from typing import Dict, List, Tuple



class EarlyStopping:
    """
    Early stopping handler that monitors Validation Macro-F1 score.
    Saves model checkpoint when Macro-F1 achieves a new peak.
    """
    def __init__(self, patience: int = 10, min_delta: float = 1e-4, checkpoint_path: str = "best_model.pth"):
        self.patience = patience
        self.min_delta = min_delta
        self.checkpoint_path = checkpoint_path
        self.counter = 0
        self.best_score = -float('inf')
        self.early_stop = False
        self.best_model_weights = None

    def __call__(self, current_score: float, model: nn.Module) -> bool:
        """
        Returns True if early stopping should terminate training.
        """
        if current_score > self.best_score + self.min_delta:
            self.best_score = current_score
            self.counter = 0
            self.best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(self.best_model_weights, self.checkpoint_path)
            print(f"[*] Validation Macro-F1 improved to {current_score:.4f}. Model weights saved to {self.checkpoint_path}")
        else:
            self.counter += 1
            print(f"[!] EarlyStopping counter: {self.counter}/{self.patience} (Best Macro-F1: {self.best_score:.4f})")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


def setup_differential_optimizer(
    model: nn.Module,
    backbone_lr: float = 1e-5,
    head_lr: float = 1e-4,
    weight_decay: float = 1e-2
) -> AdamW:
    """
    Sets up differential learning rates:
    - Backbone features (DenseNet & ViT) receive smaller LR to preserve pretrained representations.
    - CBAM attention & classification head receive larger LR to rapidly learn dermatological fusion.
    """
    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "densenet_features" in name or "vit" in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer_grouped_parameters = [
        {"params": backbone_params, "lr": backbone_lr, "weight_decay": weight_decay},
        {"params": head_params, "lr": head_lr, "weight_decay": weight_decay}
    ]

    return AdamW(optimizer_grouped_parameters)


def train_one_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    device: torch.device
) -> Tuple[float, float, float]:
    """Runs a single training epoch."""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_targets = []

    pbar = tqdm(dataloader, desc="Training Batch", leave=False)
    for images, targets, _ in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        # Automatic Mixed Precision
        with autocast(enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(targets.cpu().numpy())

        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def evaluate_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    device: torch.device
) -> Tuple[float, float, float]:
    """Evaluates validation performance."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []

    for images, targets, _ in dataloader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast(enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, targets)

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(targets.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return epoch_loss, epoch_acc, epoch_f1


def fit_model(
    model: nn.Module,
    train_loader,
    val_loader,
    class_weights: torch.Tensor,
    epochs: int = 50,
    backbone_lr: float = 1e-5,
    head_lr: float = 1e-4,
    patience: int = 10,
    checkpoint_path: str = "best_model.pth"
) -> Dict[str, List[float]]:
    """
    Full training routine orchestrating AdamW, CosineAnnealingLR, Focal Loss, 
    and Validation Macro-F1 early stopping.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n[Training] Initializing training on target device: {device}")
    model = model.to(device)

    # Multi-Class Focal Loss with computed inverse class weights
    criterion = FocalLoss(alpha=class_weights.to(device), gamma=2.0)
    
    # Differential AdamW Optimizer
    optimizer = setup_differential_optimizer(
        model, backbone_lr=backbone_lr, head_lr=head_lr, weight_decay=1e-2
    )

    # Cosine Annealing Learning Rate Scheduler
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    # AMP Gradient Scaler
    scaler = GradScaler(enabled=torch.cuda.is_available())
    
    # Early Stopping based on Macro-F1
    early_stopping = EarlyStopping(patience=patience, checkpoint_path=checkpoint_path)

    history = {
        "train_loss": [], "train_acc": [], "train_macro_f1": [],
        "val_loss": [], "val_acc": [], "val_macro_f1": [], "lr": []
    }

    start_time = time.time()
    for epoch in range(1, epochs + 1):
        print(f"\n--- Epoch {epoch}/{epochs} ---")
        
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device
        )
        val_loss, val_acc, val_f1 = evaluate_epoch(
            model, val_loader, criterion, device
        )
        
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_macro_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_macro_f1"].append(val_f1)
        history["lr"].append(current_lr)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Train Macro-F1: {train_f1:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}% | Val Macro-F1:   {val_f1:.4f} | LR: {current_lr:.6f}")

        # Check early stopping condition
        if early_stopping(val_f1, model):
            print(f"\n[Training] Early stopping triggered at epoch {epoch}.")
            break

    total_time = time.time() - start_time
    print(f"\n[Training] Finished in {total_time/60:.2f} minutes.")
    print(f"[Training] Peak Validation Macro-F1: {early_stopping.best_score:.4f}")

    # Load best checkpoint weights before returning
    if early_stopping.best_model_weights is not None:
        model.load_state_dict(early_stopping.best_model_weights)
        print("[Training] Best model weights restored to memory.")

    return history


---
## 📊 Block 6: Evaluation (5-Fold TTA + MC Dropout) & Grad-CAM Explainability
- **5-Fold Test-Time Augmentation (TTA):** Eliminates boundary variance and maximizes diagnostic accuracy.
- **Monte Carlo Dropout Uncertainty:** Samples 10 stochastic forward passes to quantify epistemic uncertainty.
- **Grad-CAM:** Generates high-resolution heatmaps overlaid on dermoscopy images.


In [ ]:
"""
=============================================================================
Phase 4 & 5: Inference, Test-Time Augmentation (TTA), Monte Carlo Uncertainty
             and Comprehensive Evaluation
=============================================================================
Author: Lead AI Computer Vision Engineer
Project: SOTA Skin Lesion Classification (HAM10000)

Techniques Implemented:
-----------------------------------------------------------------------------
1. Test-Time Augmentation (TTA):
   - 5-fold spatial transformations (Original, H-Flip, V-Flip, Rot90, Rot270)
   - Averages predictive probabilities to smooth decision boundaries and 
     boost final evaluation accuracy.
2. Monte Carlo (MC) Dropout:
   - Activates dropout modules during inference across 10 stochastic passes.
   - Quantifies prediction variance & epistemic uncertainty for clinical safety.
3. Diagnostic Metrics Suite:
   - Accuracy, Macro-F1, Weighted-F1, Sensitivity (Recall), Specificity.
   - Multi-class Confusion Matrix with normalized counts.
=============================================================================
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from typing import Dict, Tuple, List


def enable_mc_dropout(model: nn.Module):
    """
    Sets Dropout layers to train mode while keeping BatchNorm/LayerNorm in eval mode.
    This enables Monte Carlo sampling of model parameter distributions.
    """
    model.eval()
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)):
            m.train()


@torch.no_grad()
def predict_with_tta(model: nn.Module, image_tensor: torch.Tensor, device: torch.device) -> torch.Tensor:
    """
    Applies Test-Time Augmentation (TTA) with 5 spatial variants.
    
    Variants:
    1. Original: x
    2. Horizontal Flip: torch.flip(x, dims=[3])
    3. Vertical Flip: torch.flip(x, dims=[2])
    4. 90-degree Rotation: torch.rot90(x, 1, dims=[2, 3])
    5. 270-degree Rotation: torch.rot90(x, 3, dims=[2, 3])
    
    Args:
        model: Trained PyTorch model.
        image_tensor: Batch of images [B, 3, 224, 224].
        device: Device to run computation on.
        
    Returns:
        torch.Tensor: Averaged Softmax probability tensor [B, num_classes].
    """
    model.eval()
    image_tensor = image_tensor.to(device)
    
    variants = [
        image_tensor,
        torch.flip(image_tensor, dims=[3]),
        torch.flip(image_tensor, dims=[2]),
        torch.rot90(image_tensor, 1, dims=[2, 3]),
        torch.rot90(image_tensor, 3, dims=[2, 3])
    ]
    
    probs_list = []
    for var in variants:
        logits = model(var)
        probs = F.softmax(logits, dim=1)
        probs_list.append(probs)
        
    # Average predicted probabilities across 5 variants
    avg_probs = torch.stack(probs_list, dim=0).mean(dim=0)
    return avg_probs


@torch.no_grad()
def predict_with_mc_dropout(
    model: nn.Module,
    image_tensor: torch.Tensor,
    device: torch.device,
    num_passes: int = 10
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Performs Monte Carlo (MC) Dropout across `num_passes` stochastic forward passes.
    Strictly restores deterministic model.eval() upon completion.
    
    Args:
        model: Trained model.
        image_tensor: Batch of images [B, 3, 224, 224].
        device: Target device.
        num_passes: Number of stochastic passes (default: 10).
        
    Returns:
        tuple: (mean_probabilities [B, C], prediction_variance [B, C], uncertainty_score [B])
    """
    try:
        enable_mc_dropout(model)
        image_tensor = image_tensor.to(device)
        
        stochastic_probs = []
        with torch.no_grad():
            for _ in range(num_passes):
                logits = model(image_tensor)
                probs = F.softmax(logits, dim=1)
                stochastic_probs.append(probs)
            
        # Shape: [num_passes, B, C]
        stochastic_probs = torch.stack(stochastic_probs, dim=0)
        
        # 1. Mean probability: E[p]
        mean_probs = stochastic_probs.mean(dim=0)
        
        # 2. Predictive Variance: Var(p) = E[(p - E[p])^2]
        pred_variance = stochastic_probs.var(dim=0)
        
        # 3. Scalar Epistemic Uncertainty per sample: Mean variance across classes
        uncertainty_score = pred_variance.mean(dim=1)
        
        return mean_probs, pred_variance, uncertainty_score
    finally:
        # Strictly restore model to deterministic evaluation mode
        model.eval()


@torch.no_grad()
def evaluate_test_set(
    model: nn.Module,
    test_loader,
    device: torch.device,
    use_tta: bool = True,
    use_mc_dropout: bool = True,
    mc_passes: int = 10
) -> Dict[str, any]:
    """
    Evaluates model on the unseen test dataset with TTA and Monte Carlo Uncertainty Estimation.
    Strictly enforces model.eval() and torch.no_grad() throughout evaluation.
    """
    print(f"\n[Evaluation] Evaluating Test Set (TTA: {use_tta}, MC-Dropout Passes: {mc_passes if use_mc_dropout else 0})...")
    model.eval()

    all_targets = []
    all_preds = []
    all_probs = []
    all_uncertainties = []

    with torch.no_grad():
        for batch in test_loader:
            images = batch[0]
            targets = batch[1]
            targets_np = targets.numpy()
            all_targets.extend(targets_np)

            # Enforce deterministic eval mode before computing main prediction probabilities
            model.eval()

            if use_tta:
                # 5-fold Test-Time Augmentation
                probs = predict_with_tta(model, images, device)
            else:
                images = images.to(device)
                logits = model(images)
                probs = F.softmax(logits, dim=1)

            if use_mc_dropout:
                # Compute epistemic uncertainty via MC-Dropout
                _, _, uncertainties = predict_with_mc_dropout(
                    model, images, device, num_passes=mc_passes
                )
                all_uncertainties.extend(uncertainties.cpu().numpy())
                # Restore deterministic eval mode immediately
                model.eval()

            preds = torch.argmax(probs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_probs.extend(probs.cpu().numpy())

    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    if len(all_uncertainties) > 0:
        all_uncertainties = np.array(all_uncertainties)

    # Compute Global Performance Metrics
    accuracy = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_targets, all_preds, average='weighted', zero_division=0)

    # Per-Class Sensitivity (Recall) and Specificity
    target_names = [IDX_TO_LABEL[i] for i in range(len(IDX_TO_LABEL))]
    cm = confusion_matrix(all_targets, all_preds)
    
    class_metrics = {}
    for i, name in enumerate(target_names):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - (tp + fn + fp)

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        
        class_metrics[name] = {
            "Sensitivity (Recall)": sensitivity,
            "Specificity": specificity,
            "Precision": precision,
            "Support": int(tp + fn)
        }

    macro_precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    macro_recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)

    results = {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "class_metrics": class_metrics,
        "confusion_matrix": cm,
        "targets": all_targets,
        "preds": all_preds,
        "probs": all_probs,
        "uncertainties": all_uncertainties
    }

    # Print Clinical Diagnostic Summary
    print("\n" + "="*75)
    print("                SOTA SKIN LESION TEST EVALUATION REPORT")
    print("="*75)
    print(f"Overall Accuracy:  {accuracy*100:.2f}%")
    print(f"Macro F1-Score:    {macro_f1:.4f}")
    print(f"Weighted F1-Score: {weighted_f1:.4f}")
    if len(all_uncertainties) > 0:
        print(f"Average Epistemic Uncertainty (Variance): {np.mean(all_uncertainties):.6f}")
    print("-"*75)
    print(f"{'Class':<8} | {'Sensitivity':<12} | {'Specificity':<12} | {'Precision':<10} | {'Support':<8} | {'Condition'}")
    print("-"*75)
    for name in target_names:
        m = class_metrics[name]
        full_name = CLASS_FULL_NAMES.get(name, name)
        print(f"{name:<8} | {m['Sensitivity (Recall)']*100:>10.2f}% | {m['Specificity']*100:>10.2f}% | {m['Precision']*100:>8.2f}% | {m['Support']:<8} | {full_name}")
    print("="*75)
    
    return results


def plot_confusion_matrix(cm: np.ndarray, save_path: str = "confusion_matrix.png"):
    """
    Renders and saves a normalized confusion matrix heatmap.
    """
    target_names = [IDX_TO_LABEL[i] for i in range(len(IDX_TO_LABEL))]
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt='.2%',
        cmap='Blues',
        xticklabels=target_names,
        yticklabels=target_names,
        cbar_kws={'label': 'Proportion'}
    )
    plt.title("Normalized Confusion Matrix - '3-Brain' Hybrid Architecture", fontsize=14, pad=15)
    plt.xlabel("Predicted Diagnostic Class", fontsize=12)
    plt.ylabel("Ground Truth Lesion Class", fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"[*] Confusion matrix saved to: {save_path}")

"""
=============================================================================
Phase 5: Explainable AI (XAI) - Grad-CAM Implementation for Hybrid Model
=============================================================================
Author: Lead AI Computer Vision Engineer
Project: SOTA Skin Lesion Classification (HAM10000)

Gradient-Weighted Class Activation Mapping (Grad-CAM):
-----------------------------------------------------------------------------
1. Hooks forward activations A^k and backward gradients d(y_c)/d(A^k) from the
   deepest convolutional layer of DenseNet-201 / CBAM block.
2. Computes importance weights alpha_k via Global Average Pooling of gradients:
       alpha_k^c = (1 / Z) * sum_i sum_j (d y_c / d A_{i,j}^k)
3. Computes linear combination followed by ReLU:
       L_{Grad-CAM}^c = ReLU( sum_k alpha_k^c * A^k )
4. Upsamples heatmap to (224x224) and overlays onto the preprocessed lesion image 
   using a JET colormap to illuminate dermoscopic diagnostic structures.
=============================================================================
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple


class GradCAM:
    """
    Gradient-weighted Class Activation Mapping (Grad-CAM) tailored for 
    the 3-Brain Hybrid Architecture.
    """
    def __init__(self, model: nn.Module, target_layer: Optional[nn.Module] = None):
        """
        Args:
            model (nn.Module): The 3-Brain Hybrid Model.
            target_layer (nn.Module, optional): Target convolutional layer.
                                               Defaults to DenseNet201 norm5.
        """
        self.model = model
        self.model.eval()

        # Default to the final batchnorm/conv layer of DenseNet-201
        if target_layer is None:
            self.target_layer = self.model.densenet_features[-1]  # norm5 layer
        else:
            self.target_layer = target_layer

        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        """Registers forward and backward hooks on the target layer."""
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate_heatmap(
        self,
        image_tensor: torch.Tensor,
        target_class: Optional[int] = None
    ) -> Tuple[np.ndarray, int, float]:
        """
        Computes Grad-CAM heatmap for a single image tensor.
        
        Args:
            image_tensor (torch.Tensor): Preprocessed normalized tensor [1, 3, 224, 224].
            target_class (int, optional): Target class index. If None, uses top prediction.
            
        Returns:
            tuple: (heatmap [224, 224] in [0, 1], predicted_class_idx, prediction_confidence)
        """
        self.model.zero_grad()
        
        # Forward pass
        logits = self.model(image_tensor)
        probs = F.softmax(logits, dim=1)
        
        if target_class is None:
            target_class = torch.argmax(logits, dim=1).item()
            
        confidence = probs[0, target_class].item()
        
        # Backward pass for the score of the target class
        score = logits[0, target_class]
        score.backward(retain_graph=True)
        
        # Gradients: [1, C, H, W] -> Global Average Pooling -> [C]
        gradients = self.gradients[0]  # [C, H, W]
        activations = self.activations[0]  # [C, H, W]
        
        weights = torch.mean(gradients, dim=(1, 2))  # [C]
        
        # Weighted combination of activation channels
        cam = torch.zeros(activations.shape[1:], dtype=torch.float32, device=activations.device)
        for i, w in enumerate(weights):
            cam += w * activations[i]
            
        # Apply ReLU to isolate features with a POSITIVE contribution
        cam = F.relu(cam)
        cam = cam.cpu().numpy()
        
        # Normalize between [0, 1]
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam)
            
        # Resize to original input spatial dimensions (224, 224)
        cam_resized = cv2.resize(cam, (image_tensor.shape[3], image_tensor.shape[2]), interpolation=cv2.INTER_CUBIC)
        cam_resized = np.clip(cam_resized, 0.0, 1.0)
        return cam_resized, target_class, confidence


def overlay_cam_on_image(
    image: np.ndarray,
    cam: np.ndarray,
    alpha: float = 0.5,
    colormap: int = cv2.COLORMAP_JET
) -> np.ndarray:
    """
    Overlays Grad-CAM heatmap on RGB image.
    
    Args:
        image (np.ndarray): RGB image in [0, 255] uint8.
        cam (np.ndarray): Heatmap in [0, 1] float32.
        alpha (float): Transparency blending weight.
        colormap (int): OpenCV colormap.
        
    Returns:
        np.ndarray: Blended RGB image.
    """
    heatmap = np.uint8(255 * cam)
    heatmap_colored = cv2.applyColorMap(heatmap, colormap)
    heatmap_rgb = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    
    overlay = np.float32(heatmap_rgb) * alpha + np.float32(image) * (1.0 - alpha)
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    return overlay


def visualize_gradcam_diagnosis(
    image_preprocessed: np.ndarray,
    cam: np.ndarray,
    predicted_class: int,
    confidence: float,
    ground_truth: Optional[int] = None,
    save_path: Optional[str] = "gradcam_diagnosis.png"
):
    """
    Visualizes raw preprocessed lesion, attention heatmap, and overlay side-by-side.
    """
    overlay = overlay_cam_on_image(image_preprocessed, cam, alpha=0.5)
    pred_label = IDX_TO_LABEL.get(predicted_class, str(predicted_class))
    pred_full = CLASS_FULL_NAMES.get(pred_label, pred_label)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. Clean Preprocessed Lesion
    axes[0].imshow(image_preprocessed)
    title_0 = "Preprocessed (DullRazor+CLAHE)"
    if ground_truth is not None:
        gt_label = IDX_TO_LABEL.get(ground_truth, str(ground_truth))
        title_0 += f"\nTrue: {gt_label}"
    axes[0].set_title(title_0, fontsize=11)
    axes[0].axis('off')

    # 2. Grad-CAM Heatmap
    im2 = axes[1].imshow(cam, cmap='jet')
    axes[1].set_title("Grad-CAM Attention Map", fontsize=11)
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

    # 3. Clinical Overlay
    axes[2].imshow(overlay)
    axes[2].set_title(f"Clinical Overlay\nPred: {pred_label} ({confidence*100:.1f}%)\n{pred_full}", fontsize=11)
    axes[2].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"[*] Grad-CAM diagnostic visualization saved to: {save_path}")
    plt.close()


---
## 🚀 Block 7: Complete End-to-End Real Training & Evaluation
Executes the full pipeline on real HAM10000 images, evaluates on unseen test lesions, and exports `best_model.pth`.


In [ ]:
# ===========================================================================
# Full End-to-End Execution on Genuine HAM10000 Dataset
# ===========================================================================
DATA_DIR = "./data/ham10000"
os.makedirs(DATA_DIR, exist_ok=True)

# 1. Download full dataset
print("[*] Downloading / Resolving HAM10000 Dataset...")
dataset_path = download_and_extract_ham10000(DATA_DIR)

# 2. Build metadata DataFrame with zero-leakage patient-level partition
df = prepare_ham10000_dataframe(dataset_path)
print(f"Total dataset indexed: {len(df)} images across {df['lesion_id'].nunique()} unique lesions.")

train_df, val_df, test_df = create_stratified_splits(df, val_ratio=0.15, test_ratio=0.15, random_state=42)
print(f"Partition: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)} images.")

# 3. Create DataLoaders
train_loader, val_loader, test_loader, class_weights = build_dataloaders(
    train_df, val_df, test_df,
    batch_size=32 if torch.cuda.is_available() else 16,
    num_workers=2 if torch.cuda.is_available() else 0,
    img_size=224,
    use_weighted_sampler=True
)

# 4. Instantiate 3-Brain Model
print("[*] Instantiating 3-Brain Hybrid Model...")
model = build_model(num_classes=7, pretrained=True, dropout_rate=0.4).to(device)

# 5. Train with Focal Loss and Cosine Annealing (Monitored by Macro-F1)
print("[*] Beginning Genuine Training across HAM10000...")
history = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights,
    epochs=40,
    backbone_lr=1e-5,
    head_lr=1e-4,
    patience=8,
    checkpoint_path="best_model.pth"
)

# 6. Evaluate on Held-Out Test Partition with 5-Fold TTA & Monte Carlo Dropout
print("\n" + "="*80)
print(" FINAL HELD-OUT TEST EVALUATION (5-Fold TTA + MC Dropout)")
print("="*80)
results = evaluate_test_set(
    model=model,
    test_loader=test_loader,
    device=device,
    use_tta=True,
    use_mc_dropout=True,
    mc_passes=10
)

# Plot & Save Confusion Matrix
plot_confusion_matrix(results["confusion_matrix"], save_path="confusion_matrix.png")
print(f"\n[+] Real Test Accuracy: {results['accuracy']*100:.2f}%")
print(f"[+] Real Macro F1-Score: {results['macro_f1']:.4f}")

# 7. Generate Grad-CAM Diagnostic Visualization
print("\n[*] Generating Grad-CAM Diagnostic Visualization on Test Lesion...")
sample_images, sample_targets, sample_ids = next(iter(test_loader))
sample_img = sample_images[0:1].to(device)

gradcam_engine = GradCAM(model)
heatmap, pred_class, confidence = gradcam_engine.generate_heatmap(sample_img)

# Unnormalize for display
img_vis = sample_img[0].cpu().numpy().transpose(1, 2, 0)
img_vis = (img_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
img_vis = np.clip(img_vis * 255.0, 0, 255).astype(np.uint8)

visualize_gradcam_diagnosis(
    image_preprocessed=img_vis,
    cam=heatmap,
    predicted_class=pred_class,
    confidence=confidence,
    ground_truth=sample_targets[0].item(),
    save_path="gradcam_diagnosis.png"
)

# 8. Download Checkpoint to Local Machine (Colab Helper)
try:
    from google.colab import files
    print("\n[+] Triggering browser download of genuine 'best_model.pth' and 'confusion_matrix.png'...")
    files.download('best_model.pth')
    files.download('confusion_matrix.png')
except Exception as e:
    print(f"Run outside Colab or files.download skipped: {e}")
